# Experiments

Lightweight, CSV + filesystem-based experiment tracking.

Each run writes one row to `experiments/runs.csv` (the master index) plus a
folder under `experiments/runs/{run_id}/` containing the heavy artifacts
(params, per-fold models, OOF probabilities, feature importance, env snapshot,
git diff, notes). Everything is inspectable with any text editor — no MLflow /
W&B / DVC required.

## 1. Imports and paths

Standard library + a handful of modelling imports. The `DATA_DIR` points at the
preprocessed dataframes saved by `Baselines.ipynb`; paths under `EXPERIMENTS_DIR`
are created lazily on the first `save_run` call.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import hashlib
import subprocess
import platform
import sys
import uuid
import time
import datetime
from pathlib import Path
import types as _types

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

PROJECT_ROOT    = Path('.').resolve()
EXPERIMENTS_DIR = PROJECT_ROOT / 'experiments'
RUNS_CSV        = EXPERIMENTS_DIR / 'runs.csv'
RUNS_DIR        = EXPERIMENTS_DIR / 'runs'
DATA_DIR        = PROJECT_ROOT / 'data' / 'processed'
UV_LOCK         = PROJECT_ROOT / 'uv.lock'

# XGBoost ≥3.2 probes pd.util.version.Version to detect pandas ≥2.1, but
# pd.util was removed in pandas 3.0. Supply a minimal shim so the check works.
if not hasattr(pd, 'util'):
    from packaging.version import Version as _Version
    pd.util = _types.SimpleNamespace(
        version=_types.SimpleNamespace(Version=_Version)
    )

## 2. Utility functions

One markdown + code pair per utility, each justified by a single property of a
run we need to record. All utilities are pure functions; `save_run` is the only
one that touches the filesystem.

### `hash_file` — data version fingerprint

A short SHA-256 prefix of the input parquet's bytes. Storing this in the CSV row
answers "did this run use the same input data as that one?" in O(1). Truncating
to 12 hex chars is enough to make accidental collisions astronomically unlikely
for a handful of data files, while staying readable in a table.

In [31]:
def hash_file(path: Path, n: int = 12) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()[:n]

### `git_info` — code state

Returns the short commit hash, a `dirty` flag, and the full `git diff` patch of
uncommitted changes. The hash alone is insufficient whenever the working tree is
dirty — the patch makes those runs reconstructible anyway (apply the patch on top
of the commit and you get the exact code that ran).

Uses `subprocess.run(..., check=False)` so the function degrades gracefully if
the notebook is ever run outside a git repo.

In [32]:
def git_info() -> dict:
    def _run(args: list[str]) -> str:
        r = subprocess.run(['git', *args], capture_output=True, text=True, check=False)
        return r.stdout.strip() if r.returncode == 0 else ''

    short_hash = _run(['rev-parse', '--short', 'HEAD'])
    diff_text  = _run(['diff', 'HEAD'])
    return {
        'hash':  short_hash,
        'dirty': bool(diff_text),
        'diff':  diff_text,
    }

### `environment_info` — Python / OS / lockfile

Captures the Python interpreter version, the platform string, and the full contents
of `uv.lock` (the authoritative dependency record for this project). The lockfile
is too large to embed in the CSV, so `save_run` writes it to `environment.txt`
inside the run directory and only the version + platform land in the CSV row.

In [33]:
def environment_info() -> dict:
    uv_lock_contents = UV_LOCK.read_text(encoding='utf-8') if UV_LOCK.exists() else ''
    return {
        'python_version':   sys.version.split()[0],
        'platform':         platform.platform(),
        'uv_lock_contents': uv_lock_contents,
    }

### `new_run_id` and `params_hash` — identifiers

- **`new_run_id`**: `{YYYYMMDD-HHMMSS}-{uuid6}`. The timestamp orders runs
  chronologically; the short UUID tail defends against same-second collisions
  during Optuna-style sweeps.
- **`params_hash`**: 8-char SHA-256 of the sorted-JSON representation of the
  params dict. Makes "have I already tried this exact config?" a one-liner
  (`(runs['params_hash'] == h).any()`). The sort ensures dict-order differences
  don't produce spurious mismatches.

In [34]:
def new_run_id() -> str:
    ts = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
    return f'{ts}-{uuid.uuid4().hex[:6]}'


def params_hash(params: dict, n: int = 8) -> str:
    payload = json.dumps(params, sort_keys=True, default=str).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()[:n]

### `save_run` — the workhorse

Persists one run in two places:
1. **`experiments/runs/{run_id}/`** — heavy artifacts (params, metrics, OOF,
   test predictions, feature importance, uv.lock copy, git diff, notes, fold
   models). Arrays go to `.npy`; structured data to JSON; text to plain files.
2. **`experiments/runs.csv`** — one scalar summary row. Header is written only
   when the file doesn't yet exist, so repeated calls just append.

`save_models=False` can be used to skip the `joblib.dump` of per-fold models
(each LGBM / XGB / CatBoost booster can easily be 20–50 MB × n_splits).

In [ ]:
def save_run(run_dict: dict, artifacts: dict, run_id: str) -> Path:
    run_dir = RUNS_DIR / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    # JSON artifacts
    with open(run_dir / 'params.json', 'w', encoding='utf-8') as f:
        json.dump(artifacts['params'], f, indent=2, default=str)
    with open(run_dir / 'metrics.json', 'w', encoding='utf-8') as f:
        json.dump(artifacts['metrics'], f, indent=2, default=str)

    # Array artifacts
    np.save(run_dir / 'oof_proba.npy',        artifacts['oof_proba'])
    np.save(run_dir / 'test_proba_folds.npy', artifacts['test_proba_folds'])
    np.save(run_dir / 'test_proba_mean.npy',  artifacts['test_proba_mean'])

    # Feature importance (optional)
    if artifacts.get('feature_importance') is not None:
        artifacts['feature_importance'].to_csv(
            run_dir / 'feature_importance.csv', index=False,
        )

    # Text artifacts
    (run_dir / 'environment.txt').write_text(artifacts['environment_text'], encoding='utf-8')
    (run_dir / 'git_diff.patch').write_text(artifacts['git_diff'],          encoding='utf-8')
    (run_dir / 'notes.md').write_text(artifacts['notes'],                   encoding='utf-8')

    # Fold models (optional)
    if artifacts.get('models'):
        models_dir = run_dir / 'models'
        models_dir.mkdir(exist_ok=True)
        for i, model in enumerate(artifacts['models']):
            joblib.dump(model, models_dir / f'fold_{i}.joblib')

    # Append to master CSV. Fast path is plain append; if run_dict introduces
    # new columns vs. the existing header (e.g. data_version added after older
    # runs were logged), fall back to read → outer-join → rewrite so the file
    # stays well-formed and old rows get NaN in the new columns.
    EXPERIMENTS_DIR.mkdir(exist_ok=True)
    new_row = pd.DataFrame([run_dict])
    if RUNS_CSV.exists():
        existing_cols = list(pd.read_csv(RUNS_CSV, nrows=0).columns)
        if existing_cols == list(new_row.columns):
            new_row.to_csv(RUNS_CSV, mode='a', header=False, index=False)
        else:
            existing_full = pd.read_csv(RUNS_CSV)
            combined = pd.concat([existing_full, new_row], ignore_index=True)
            combined.to_csv(RUNS_CSV, index=False)
    else:
        new_row.to_csv(RUNS_CSV, index=False)

    return run_dir

### `load_runs` — the leaderboard view

Reads `runs.csv` into a DataFrame sorted by `oof_score` descending. Returns an
empty frame if the CSV doesn't exist yet, so the display cells don't error before
the first run.

In [36]:
def load_runs() -> pd.DataFrame:
    if not RUNS_CSV.exists():
        return pd.DataFrame()
    df = pd.read_csv(RUNS_CSV)
    if 'oof_score' in df.columns:
        df = df.sort_values('oof_score', ascending=False).reset_index(drop=True)
    return df

## 3. Training-loop template

The cells below are the ones you **copy and edit** per experiment. The two
edit points are:

1. **`engineer_features(df)`** — row-wise feature engineering, cached to a
   versioned parquet so future runs reload instantly.
2. **`run_config`** — model factory, params, tag, notes, CV config, metric,
   and the `data_version` label tying the run back to a specific FE recipe.

Everything else — the fold loop, OOF accumulation, per-fold test predictions,
feature-importance aggregation, metadata collection, and the final `save_run`
call — stays the same across experiments.

In [ ]:
# --- Load preprocessed data ---
train_df = pd.read_parquet(DATA_DIR / 'train_df.parquet')
test_df  = pd.read_parquet(DATA_DIR / 'test_df.parquet')
print(f'Loaded raw: train_df {train_df.shape}, test_df {test_df.shape}')

### Feature engineering

`engineer_features(df)` is the per-experiment hook for adding features. The
output is cached to `data/processed/train_df_{DATA_VERSION}.parquet` so future
runs reload it instantly — and so any past run is reconstructible by reading
the parquet whose suffix matches that run's `data_version` field.

**Strict no-leakage rule**: only row-wise (stateless) transforms here. Anything
that needs to be *fitted* on training data — target encoding, scaling,
imputation, frequency counts, or any aggregate over rows — would leak val/test
information across folds if computed once on the full training set. Those
must be done inside the fold loop instead.

When you change `engineer_features`, **bump `DATA_VERSION`** so a new parquet
is written rather than the stale cache being reused.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

DATA_VERSION = 'fe_v0'  # bump whenever engineer_features changes — see rule below


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Row-wise (stateless) feature engineering.

    Each new column must depend ONLY on values from the row being transformed.
    No aggregates, no fitted encoders, no cross-row statistics — those leak
    val/test information into training when applied before the CV split, so
    they belong inside the fold loop instead.

    Examples of acceptable transforms:
        df['x_per_y']     = df['x'] / (df['y'] + 1)
        df['log_x']       = np.log1p(df['x'])
        df['x_above_med'] = (df['x'] > 100).astype(int)
    """
    df = df.copy()
    # Add row-wise features here.
    return df


# Cache engineered parquets per DATA_VERSION. Past runs are reconstructible by
# loading the parquet whose suffix matches that run's logged data_version.
fe_train_path = DATA_DIR / f'train_df_{DATA_VERSION}.parquet'
fe_test_path  = DATA_DIR / f'test_df_{DATA_VERSION}.parquet'

if fe_train_path.exists() and fe_test_path.exists():
    train_df = pd.read_parquet(fe_train_path)
    test_df  = pd.read_parquet(fe_test_path)
    print(f'Loaded cached FE: {DATA_VERSION}')
else:
    train_df = engineer_features(train_df)
    test_df  = engineer_features(test_df)
    pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False), fe_train_path)
    pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False), fe_test_path)
    print(f'Computed and cached FE: {DATA_VERSION}')

# Refresh feature list and design matrices from the engineered dataframes.
encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}  features: {len(encoded_features)}')

### Run configuration

Per-experiment config — model factory, params, metric, CV splitter, free-text
tag/notes. `data_version` is pulled from the FE cell above so the run row
links back to the parquet it was trained on.

In [ ]:
from sklearn.base import clone

# --- Run config (edit this per experiment) ---
lr_pipeline_fitted = joblib.load(PROJECT_ROOT / 'data' / 'models' / 'lr_pipeline.pkl')

run_config = {
    'model_factory': lambda params: clone(lr_pipeline_fitted),
    'params':        lr_pipeline_fitted.get_params(),
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'lr-pipeline',
    'notes':         'Logistic regression pipeline from Baselines.ipynb.',
    'parent_run_id': '',
    'save_models':   True,
    'data_version':  DATA_VERSION,
}

In [ ]:

# --- Load preprocessed data ---
train_df = pd.read_parquet(DATA_DIR / 'train_df.parquet')
test_df  = pd.read_parquet(DATA_DIR / 'test_df.parquet')

encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]


In [ ]:

# --- Run config (edit this per experiment) ---
with open(PROJECT_ROOT / 'data' / 'models' / 'catboost_best_params.json') as f:
    catboost_params = json.load(f)

run_config = {
    'model_factory': lambda params: CatBoostClassifier(**params),
    'params':        catboost_params,
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'catboost-best-params',
    'notes':         'CatBoost with Optuna best params from Baselines.ipynb.',
    'parent_run_id': '',
    'save_models':   True,
}

In [ ]:
# =============================================================================
# 1. INITIALISE — accumulator arrays and per-fold trackers
# =============================================================================
# We allocate everything up-front so the fold loop is a pure write-into-slot
# operation (no list.append for the heavy arrays). One row per CV fold, one
# column per training sample / test sample as appropriate.

run_id = new_run_id()
print(f'Run ID: {run_id}')
print(f'Tag:    {run_config["tag"]}')
print()

cv = run_config['cv']

# OOF (out-of-fold) probabilities: each training row gets exactly one prediction,
# made by the model that did NOT see it during fit. Concatenating the per-fold
# val-set predictions reconstructs a full-length vector that is honest about
# generalisation error — this is what `oof_score` is computed on later.
oof_proba = np.zeros(len(X_train))

# Per-fold test predictions: shape (n_test, n_splits). We average across the
# fold axis at the end ("bag of folds") to get smoother test probabilities than
# any single fold's model would produce.
test_proba_folds = np.zeros((len(X_test), cv.n_splits))

# Lightweight per-fold collectors. fold_models is kept around so we can dump
# them at the end and so `sample_model.get_params()` works after the loop.
fold_scores, fold_times, fold_models = [], [], []
gain_imps,   split_imps               = [], []  # tree-model importances; stays empty for non-tree models

# =============================================================================
# 2. FOLD LOOP — fit → predict val → predict test → record
# =============================================================================
# StratifiedKFold guarantees each fold has the same Churn class balance as the
# full training set. tr_idx / va_idx are positional indices into X_train.

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train)):
    # Fresh model per fold. model_factory is a closure over run_config['params']
    # — for sklearn pipelines it's `clone(...)`; for boosters it's `Cls(**p)`.
    model = run_config['model_factory'](run_config['params'])

    # Time only fit (not predict) — fit dominates wall time and is what we
    # benchmark against when comparing models.
    t0 = time.perf_counter()
    model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    elapsed = time.perf_counter() - t0

    # Validation predictions land in their original positions in oof_proba.
    # After all folds, every training row has been predicted exactly once.
    va_proba = model.predict_proba(X_train.iloc[va_idx])[:, 1]
    oof_proba[va_idx] = va_proba

    # Test predictions: each fold predicts the FULL test set; we keep them
    # separate so the user can later inspect fold-level disagreement, and so
    # the bagged mean is just `.mean(axis=1)`.
    test_proba_folds[:, fold] = model.predict_proba(X_test)[:, 1]

    # Threshold at 0.5 because run_config['metric'] is accuracy_score, which
    # expects labels not probabilities. (Swap to roc_auc_score and you'd pass
    # va_proba directly — that's a future-experiment knob.)
    score = run_config['metric'](y_train.iloc[va_idx], (va_proba >= 0.5).astype(int))
    fold_scores.append(score)
    fold_times.append(elapsed)
    fold_models.append(model)

    # Feature importance is LightGBM-flavoured here (gain + split). Trees
    # expose `.booster_`; sklearn pipelines and XGBoost's sklearn wrapper
    # don't, so this branch is silently skipped for them.
    if hasattr(model, 'booster_'):
        gain_imps.append(model.booster_.feature_importance(importance_type='gain'))
        split_imps.append(model.booster_.feature_importance(importance_type='split'))

    print(f'Fold {fold}: {run_config["metric_name"]}={score:.4f}  (fit {elapsed:.1f}s)')

# =============================================================================
# 3. AGGREGATE — bag-of-folds test predictions, OOF score, mean importance
# =============================================================================
# test_proba_mean is the standard CV-bagging output and what you'd use as the
# Kaggle submission's probabilities for this run.
test_proba_mean = test_proba_folds.mean(axis=1)

# oof_score is computed once on the full vector — NOT the mean of fold_scores.
# The two usually agree closely, but oof_score is what most Kaggle-style
# leaderboards correlate with because it sees every training row exactly once.
oof_score = run_config['metric'](y_train, (oof_proba >= 0.5).astype(int))

# Mean gain/split across folds gives a more stable importance ranking than any
# single fold. Stays None for non-tree models — save_run handles that case.
feat_imp = None
if gain_imps:
    feat_imp = pd.DataFrame({
        'feature': encoded_features,
        'gain':    np.mean(gain_imps,  axis=0),
        'split':   np.mean(split_imps, axis=0),
    })

# =============================================================================
# 4. COLLECT METADATA — everything needed to reproduce or compare this run
# =============================================================================
# git_info captures code state (hash + dirty flag + full diff patch).
# environment_info captures Python version, OS, and the uv.lock contents.
# sample_model is just fold 0 — all folds share hyperparameters, so .get_params()
# from any of them is the canonical record.
git          = git_info()
env          = environment_info()
sample_model = fold_models[0]
full_params  = sample_model.get_params()

# run_dict is the SCALAR summary written as one row in experiments/runs.csv.
# Anything that doesn't fit a single CSV cell (full params dict, OOF array,
# git diff, uv.lock) goes into the artifacts dict instead.
run_dict = {
    'run_id':            run_id,
    'timestamp':         datetime.datetime.now().isoformat(timespec='seconds'),
    'tag':               run_config['tag'],
    'notes':             run_config['notes'],
    'status':            'success',
    'parent_run_id':     run_config['parent_run_id'],
    'git_hash':          git['hash'],
    'git_dirty':         git['dirty'],
    'data_hash':         hash_file(DATA_DIR / 'train_df.parquet'),
    'data_version':      run_config['data_version'],  # FE recipe label; pairs with data_hash to fully describe inputs
    'python_version':    env['python_version'],
    'platform':          env['platform'],
    'model_class':       type(sample_model).__name__,
    'params_hash':       params_hash(full_params),  # 8-char SHA for "have I tried this exact config?" lookups
    'cv_type':           type(cv).__name__,
    'n_splits':          cv.n_splits,
    'random_state':      cv.random_state,
    'n_train':           len(X_train),
    'n_test':            len(X_test),
    'n_features':        X_train.shape[1],
    'metric':            run_config['metric_name'],
    'fold_scores_mean':  float(np.mean(fold_scores)),
    'fold_scores_std':   float(np.std(fold_scores)),
    'oof_score':         float(oof_score),
    'training_time_sec': float(sum(fold_times)),
    'lb_public':         np.nan,  # filled manually after Kaggle submission
    'lb_private':        np.nan,
    'artifact_dir':      str((RUNS_DIR / run_id).relative_to(PROJECT_ROOT)),
}

# metrics.json — richer than the CSV row: holds per-fold breakdowns, the full
# CV config, and the feature list (so a stacking script can verify column
# alignment when loading OOF predictions from a past run).
metrics = {
    'fold_scores':       [float(s) for s in fold_scores],
    'fold_times_sec':    [float(t) for t in fold_times],
    'oof_score':         float(oof_score),
    'fold_scores_mean':  float(np.mean(fold_scores)),
    'fold_scores_std':   float(np.std(fold_scores)),
    'training_time_sec': float(sum(fold_times)),
    'cv_config': {
        'type':         type(cv).__name__,
        'n_splits':     cv.n_splits,
        'random_state': cv.random_state,
        'shuffle':      getattr(cv, 'shuffle', None),
    },
    'features': encoded_features,
}

# artifacts is the heavy stuff — arrays, DataFrames, large strings, fold models.
# save_run dispatches each key to its appropriate file format under runs/{run_id}/.
artifacts = {
    'params':             full_params,
    'metrics':            metrics,
    'oof_proba':          oof_proba,
    'test_proba_folds':   test_proba_folds,
    'test_proba_mean':    test_proba_mean,
    'feature_importance': feat_imp,
    'environment_text':   env['uv_lock_contents'],
    'git_diff':           git['diff'],
    'notes':              run_config['notes'],
    'models':             fold_models if run_config['save_models'] else None,
}

# =============================================================================
# 5. PERSIST — one CSV row + a per-run directory with all the heavy artifacts
# =============================================================================
save_run(run_dict, artifacts, run_id)
print()
print(f'OOF {run_config["metric_name"]}: {oof_score:.4f}')
print(f'Folds:        {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'Saved to:     {RUNS_DIR / run_id}')

## 4. View all runs

Reads `runs.csv` sorted by `oof_score` descending. The display below shows the
columns most useful in a leaderboard view; the full row is always available via
`load_runs()`.

In [55]:
runs = load_runs()
if runs.empty:
    print('No runs logged yet.')
else:
    display(runs[[
        'run_id', 'tag', 'model_class',
        'oof_score', 'fold_scores_mean', 'fold_scores_std',
        'lb_public', 'lb_private', 'notes',
    ]])

,run_id,tag,model_class,oof_score,fold_scores_mean,fold_scores_std,lb_public,lb_private,notes
0,20260424-195649-307853,xgb-best-params,XGBClassifier,0.861963,0.861963,0.000760,NaN,NaN,XGBoost with Optuna best params from Baselines...
1,20260424-194123-594ade,lgbm-best-params-reproduce,LGBMClassifier,0.861724,0.861724,0.000605,NaN,NaN,First logged run — should reproduce Baselines....
2,20260424-200702-c831e4,catboost-best-params,CatBoostClassifier,0.861697,0.861697,0.000672,NaN,NaN,CatBoost with Optuna best params from Baseline...
3,20260424-203302-53c36a,catboost-best-params,CatBoostClassifier,0.861697,0.861697,0.000672,NaN,NaN,CatBoost with Optuna best params from Baseline...
4,20260424-203804-5a6511,lr-pipeline,Pipeline,0.854512,0.854512,0.001006,NaN,NaN,Logistic regression pipeline from Baselines.ip...


## 5. Reloading a past run

Every artifact is retrievable by `run_id`. Common use cases:
- Load `oof_proba.npy` for stacking, threshold tuning, or calibration
- Load `params.json` to re-instantiate the model elsewhere
- `joblib.load` a fold model for inspection or to run SHAP

The cell below reloads the most recent run.

In [56]:
runs = load_runs()
if runs.empty:
    print('Log at least one run first.')
else:
    latest_run_id = runs.iloc[0]['run_id']
    run_dir = RUNS_DIR / latest_run_id

    with open(run_dir / 'params.json') as f:
        saved_params = json.load(f)

    oof = np.load(run_dir / 'oof_proba.npy')

    print(f'Reloaded run: {latest_run_id}')
    print(f'OOF shape:    {oof.shape}')
    print(f'OOF mean:     {oof.mean():.4f}')
    print(f'# params:     {len(saved_params)}')

    fold0_path = run_dir / 'models' / 'fold_0.joblib'
    if fold0_path.exists():
        m0 = joblib.load(fold0_path)
        print(f'Fold-0 model: {type(m0).__name__}')

Reloaded run: 20260424-195649-307853
OOF shape:    (594194,)
OOF mean:     0.2252
# params:     40
Fold-0 model: XGBClassifier
